In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
%cd ..

/path/to/repo/my-coles-gnn-experimetns/scenario_gender


In [6]:
import sys; import os; sys.path.append(os.path.dirname(os.getcwd()))

In [7]:
from typing import List

from omegaconf import OmegaConf 

from ptls_extension_2024_research.latex_table_creation.latex_table_creation import create_latex_table, get_metrics, get_experiment_dicts_list
from ptls_extension_2024_research.latex_table_creation.experiment_dicts_list_modifiers import bolden_top_k, sort_by_col
from ptls_extension_2024_research.latex_table_creation.prefix_map import get_idxs_where_all_metrics_superpass, prefix_map_from_idx_lst
from ptls_extension_2024_research.latex_table_creation.hyperparam_getters import get_batchsize_from_config, get_split_count_from_config, get_triplets_per_user_from_config, get_convex_loss_alpha_from_config, get_min_users_in_separated_single_bin_from_config


/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
COLES_METRICS = {r"\textbf{AUC test}": 0.877, r"\textbf{ACC test}": 0.793}

In [9]:
HYDRA_OUTPUTS_PATH ='hydra_outputs/'

In [10]:
METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME = {'auroc': r"\textbf{AUC test}", 'accuracy': r"\textbf{ACC test}"}

In [11]:
REPORT_FILE_PATH = "results/scenario_gender_bpr.txt"

In [12]:
def leave_only_experiments_from_results(expected_experiments: List[str], report_file_path) -> List[str]:
    with open(report_file_path, 'r') as f:
        report_file_content = f.read()
    
    existing_experiments = [exp_name for exp_name in expected_experiments if exp_name in report_file_content]
    return existing_experiments

In [13]:
PRETRAINED_MODELS_ROOT = 'models/pretrained_gnn_models'

In [14]:
MAX_EPOCHES=40
N_TRIPLETS_PER_ANCHOR_USER_OPTIONS=(128, 32, 8, 1)
MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS=(5,)
LOSS_ALPHA_OPTIONS=(0.5, 0.85, 0.15, 0.01, 0)


expected_experiments = []

for min_elements_in_bin_when_one_bin_only in MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS:
    for n_triplets_per_anchor_user in N_TRIPLETS_PER_ANCHOR_USER_OPTIONS:
        for loss_alpha in LOSS_ALPHA_OPTIONS:
            experiment_name = f'coles_bpr__loss_alpha_{loss_alpha}__{MAX_EPOCHES}__epoches__triplets_per_user_{n_triplets_per_anchor_user}__bin_separation_margin_{min_elements_in_bin_when_one_bin_only}__try_2'
            expected_experiments.append(experiment_name)



In [15]:
len(expected_experiments)

20

In [20]:
expected_experiments_before_clean = expected_experiments
expected_experiments = leave_only_experiments_from_results(expected_experiments, REPORT_FILE_PATH)

In [21]:
set(expected_experiments_before_clean) - set(expected_experiments)

{'coles_bpr__loss_alpha_0.01__40__epoches__triplets_per_user_8__bin_separation_margin_5__try_2',
 'coles_bpr__loss_alpha_0.15__40__epoches__triplets_per_user_8__bin_separation_margin_5__try_2',
 'coles_bpr__loss_alpha_0.5__40__epoches__triplets_per_user_8__bin_separation_margin_5__try_2',
 'coles_bpr__loss_alpha_0.85__40__epoches__triplets_per_user_8__bin_separation_margin_5__try_2',
 'coles_bpr__loss_alpha_0__40__epoches__triplets_per_user_8__bin_separation_margin_5__try_2'}

In [22]:
hyperparams_to_getters = {
    r"\textbf{Triplets per user}": get_triplets_per_user_from_config,
    r"\textbf{Loss alpha}": get_convex_loss_alpha_from_config,
    r"\textbf{Min users in separated single bin}": get_min_users_in_separated_single_bin_from_config,
}

In [23]:
experiment_dicts_list = get_experiment_dicts_list(
    expected_experiments, hyperparams_to_getters, HYDRA_OUTPUTS_PATH, 
    experiment_name_to_main_experiment_name = lambda x: x, report_file_path = REPORT_FILE_PATH,
    metric_report_name_to_metric_table_name = METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME)

In [24]:
WEIGHTS_TYPE = 'type 2'


experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
experiment_dicts_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])



EXPERIMENT_NAME = r'\makecell{Coles only with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_list
    }
}



In [26]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = "Comparison of different setups where coles was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{Triplets per user} & \textbf{Loss alpha} & \textbf{Min users in separated single bin} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{15}{*}{\makecell{Coles only with bpr}} &\multirow{15}{*}{type 2} & 128 & 0.15 & 5 & \textbf{0.883} & \textbf{0.796} \\ \cline{3-7} 

\rowcolor{gray!50}
& &128 & 0.85 & 5 & \textbf{0.881} & 0.794 \\ \cline{3-7} 

\rowcolor{gray!50}
& &32 & 0.15 & 5 & \textbf{0.881} & 0.796 \\ \cline{3-7} 

\rowcolor{gray!50}
& &1 & 0.5 & 5 & 0.88 & \textbf{0.8} \\ \cline{3-7} 

\rowcolor{gray!50}
& &32 & 0.01 & 5 & 0.878 & 0.793 \\ \cline{3-7} 

\rowcolor{gray!50}
& &32 & 0.85 & 5 & 0.877 & 0.794 \\ \cline{3-7} 
& &32 & 0.5 & 5 & 0.875 & 0.796 \\ \cline{3-7} 
& &1 & 0.15 & 5 & 0.875 & \textbf{0.797} \\ \cline{3-7} 
& &1 & 0.85 & 5 & 0.874 & 0.792 \\ \cline{3-7} 
& &128 & 0.01 & 5 & 0.873 & 0.78